# Build a basic chatbot

In this tutorial, you will build a basic chatbot. This chatbot is the basis for the following series of tutorials where you will progressively add more sophisticated capabilities, and be introduced to key LangGraph concepts along the way. Let’s dive in! 🌟

In [12]:
%%capture --no-stderr
%pip install --quiet -U langchain_aws langchain_core langgraph langgraph-prebuilt load_dotenv

In [13]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_DEFAULT_REGION"] = os.getenv('AWS_DEFAULT_REGION')

We'll use [LangSmith](https://docs.smith.langchain.com/) for [tracing](https://docs.smith.langchain.com/concepts/tracing).

In [14]:
os.environ["LANGSMITH_API_KEY"] = os.getenv('LANGSMITH_API_KEY')
os.environ["LANGSMITH_ENDPOINT"]="https://api.smith.langchain.com"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tutorial"

## Create a StateGraph

Now you can create a basic chatbot using LangGraph. This chatbot will respond directly to user messages.

Start by creating a StateGraph. A StateGraph object defines the structure of our chatbot as a "state machine". We'll add nodes to represent the llm and functions our chatbot can call and edges to specify how the bot should transition between these functions.


In [15]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

## Add a node

Next, add a "chatbot" node. Nodes represent units of work and are typically regular Python functions.

In [16]:
from langchain_aws import ChatBedrock

llm = ChatBedrock(
    model_id="mistral.mistral-7b-instruct-v0:2",
    temperature=1,
)

In [17]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}


# The first argument is the unique node name
# The second argument is the function or object that will be called whenever
# the node is used.
graph_builder.add_node("chatbot", chatbot)

Notice how the chatbot node function takes the current State as input and returns a dictionary containing an updated messages list under the key "messages". This is the basic pattern for all LangGraph node functions.

The add_messages function in our State will append the LLM's response messages to whatever messages are already in the state.

## Add an entry point

Add an entry point to tell the graph where to start its work each time it is run:

In [18]:
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

## Compile the graph

Before running the graph, we'll need to compile it. We can do so by calling compile() on the graph builder. This creates a CompiledGraph we can invoke on our state.

In [19]:
graph = graph_builder.compile()

## Visualize the graph

You can visualize the graph using the get_graph method and one of the "draw" methods, like draw_ascii or draw_png. The draw methods each require additional dependencies.

In [20]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
react_graph_memory = graph_builder.compile(checkpointer=memory)

In [21]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# Node


## Run the chatbot

Now run the chatbot!

In [22]:
def stream_graph_updates(state: MessagesState):
    result = react_graph_memory.invoke(state, config={"configurable": {"thread_id": "1"}})
    print("Assistant:", result["messages"][-1].content)


while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates({"messages": [{"role": "user", "content": user_input}]})
    except:
        # fallback if input() is not available
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates({"messages": [{"role": "user", "content": user_input}]})
        break

Assistant:  The sum of two numbers, 2 and 3, is found by adding their values together. So, the sum of 2 and 3 is:

2 + 3 = 5

Therefore, the sum of 2 and 3 is equal to 5.
Assistant:  To find the product of a sum and a number, first find the sum, then multiply the sum by the number. In this case, the sum of 2 and 3 is equal to 5, and we want to multiply this sum by 10:

5 × 10 = 50

So, the product of 5 (which is the sum of 2 and 3) and 10 is equal to 50.
Assistant:  To divide a number by another number, subtract the divisor from the dividend repeatedly until the quotient is obtained, or in this case, use long division or division by repeated subtraction. However, since we want to divide a sum by a number, first find the sum, then divide.

The sum of 2 and 3 is 5. Now, we want to divide 5 by 5:

5 ÷ 5 = 1

Therefore, the quotient of 5 (the sum) divided by 5 is equal to 1. This result implies that the sum of 2 and 3 is equivalent to 1 when divided by 5.

Keep in mind that when dealing wi